In [8]:
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

# Läs in data
df = pd.read_csv("historical_data.csv")  

target_name = "is_suspicious"

X = df.drop(columns=[target_name, "id"])
y = df[target_name]

numeric_features = [
    "day",
    "account_age_days",
    "num_prev_listings",
    "prev_reports_30d",
    "verification_level",
    "price",
    "num_images",
    "message_length",
    "contains_off_platform",
    "urgency_words",
    "payment_attempt",
    "time_to_first_response_min"
]

categorical_features = [
    "event_type",
    "category",
    "region",
    "device"
]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

print("Setup klar")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

Setup klar
X_train: (8400, 16)
X_test: (3600, 16)


## 4. Hyperparameter-tuning av vald modell

Vi valde att optimera den modell som gick vidare från modelljämförelsen. För att hålla lösningen enkel testade vi ett mindre antal hyperparametrar med GridSearchCV och 5-fold stratified cross-validation. Eftersom vårt kravkort fokuserar på att minska onödiga flaggningar valde vi att optimera mot precision.

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

pipe = Pipeline([
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

param_grid = {
    "model__C": [0.1, 1, 10],
    "model__class_weight": [None, "balanced"]
}

grid_search = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=cv,
    scoring="precision",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Bästa parametrar:", grid_search.best_params_)
print("Bästa precision (CV):", round(grid_search.best_score_, 3))

best_model = grid_search.best_estimator_

Bästa parametrar: {'model__C': 10, 'model__class_weight': None}
Bästa precision (CV): 0.595


In [10]:
results = pd.DataFrame(grid_search.cv_results_)
results = results[["params", "mean_test_score", "std_test_score"]] \
    .sort_values("mean_test_score", ascending=False)

results.head(5)

,params,mean_test_score,std_test_score
4,"{'model__C': 10, 'model__class_weight': None}",0.595455,0.113113
2,"{'model__C': 1, 'model__class_weight': None}",0.590455,0.112245
0,"{'model__C': 0.1, 'model__class_weight': None}",0.590256,0.109873
1,"{'model__C': 0.1, 'model__class_weight': 'bala...",0.192561,0.012049
3,"{'model__C': 1, 'model__class_weight': 'balanc...",0.191570,0.012161


In [11]:
test_proba = best_model.predict_proba(X_test)[:, 1]
print("Antal sannolikheter för testdata:", len(test_proba))

Antal sannolikheter för testdata: 3600


### Resultat från tuning

Grid search visade att den bästa varianten av Logistic Regression fick parametrarna **C = 10** och **class_weight = None**. Den bästa genomsnittliga precisionen i cross-validation blev **0.595**. Detta tyder på att en försiktig modell utan balanserade klassvikter passar vårt kravkort bättre, eftersom målet är att minska onödiga flaggningar.